# ED5J990H5VAZT

Packages

In [ ]:
# Imports
from foodcast.imports import *
os.chdir(find_project_root())
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, _, _, _, DATA_DIR_3_7 = DATA_DIR_3_x

# Settings
notebook_settings()  

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
loc_id = 'ED5J990H5VAZT'
df_uncleaned = load_one_res_2(loc_id)
df_targeted = pd.read_parquet(DATA_DIR_3_7 / f'{loc_id}.parquet')

In [ ]:
# Load remapping data from YAML (like loc2 does)
labeling_path = Path('scripts') / 'labeling'
remapping_path = labeling_path / 'remapping' / 'loc5_remappings.yaml'
with open(remapping_path, "r", encoding="utf-8") as f:
    remapping = yaml.load(f, Loader=yaml.FullLoader)

# Extract modification patterns from YAML
MODS_VEGAN_MEAT = remapping['modification_patterns']['MODS_VEGAN_MEAT']
MODS_MEAT = remapping['modification_patterns']['MODS_MEAT']
MODS_VEGAN_DAIRY_EGG = remapping['modification_patterns']['MODS_VEGAN_DAIRY_EGG']
MODS_DAIRY_EGG = remapping['modification_patterns']['MODS_DAIRY_EGG']
MODS_EXPLICIT_VEGAN = remapping['modification_patterns']['MODS_EXPLICIT_VEGAN']
MODS_NONE = remapping['modification_patterns']['MODS_NONE']

# Extract dish configuration from YAML
DISH_CONFIG = [(dish['dish'], dish['default_state'], dish['has_explicit_vegan_rule']) 
               for dish in remapping['dish_config']]

# Generate modification rules programmatically (keep the automation)
modification_rules_step1 = []
modification_rules_step2 = []

for base_name, final_default_state, has_explicit_vegan_rule in DISH_CONFIG:
    intermediate_name = f"Nonmeat {base_name}"
    meat_intermediate_name = f"Meat {base_name}"

    # Step 1 Rules (Base -> Intermediate)
    modification_rules_step1.extend([
        (base_name, MODS_VEGAN_MEAT, intermediate_name),
        (base_name, MODS_MEAT, meat_intermediate_name),
        (base_name, MODS_NONE, intermediate_name),
    ])

    # Step 2 Rules (Intermediate -> Final)
    vegan_final_name = f"Vegan {base_name}"
    vegetarian_final_name = f"Vegetarian {base_name}"
    meat_final_name = f"Meat {base_name}"

    if final_default_state == 'Vegan':
        default_final_name = vegan_final_name
    elif final_default_state == 'Vegetarian':
        default_final_name = vegetarian_final_name
    else:
        default_final_name = meat_final_name

    step2_rules = [
        (intermediate_name, MODS_VEGAN_DAIRY_EGG, vegan_final_name),
        (intermediate_name, MODS_DAIRY_EGG, vegetarian_final_name),
    ]

    if has_explicit_vegan_rule:
        step2_rules.append((intermediate_name, MODS_EXPLICIT_VEGAN, vegan_final_name))

    step2_rules.append((intermediate_name, MODS_NONE, default_final_name))
    modification_rules_step2.extend(step2_rules)

# Calculate rare items dynamically (items that appear less than 10 times)
rare_items = df_uncleaned['item_name'].value_counts().to_frame('counts').query('counts < 10').index.tolist()
drink_categories = remapping['drink_categories']
drink_items = df_uncleaned.query('dish_category.isin(@drink_categories)')['item_name'].value_counts().index.tolist()
items_to_remove = remapping['merch_list'] + drink_items + rare_items

df_uncleaned = remove_numbers(df_uncleaned, 'item_name')

# Step 1: Apply initial modifications
df_intermediate = fully_relabel_and_consolidate(
    df_uncleaned,
    remove=items_to_remove,
    name_changes=remapping.get("name_changes", {}),
    modification_name_changes=modification_rules_step1,
    vegan_list=remapping.get("vegan_list", []),
    vegetarian_list=remapping.get("vegetarian_list", []),
    meat_list=remapping.get("meat_list", []),
    alcohol_list=remapping.get("alcoholic_drinks", []),
    drinks_list=remapping.get("non_alcoholic_drinks", []),
    merch=remapping.get("merch_list", []),
    rare=rare_items + remapping.get("rare_list", []),
    unknown=remapping.get("unknown_list", []),
    remove_categories=drink_categories
)

# Step 2: Apply final modifications
df_relabeled = fully_relabel_and_consolidate(
    df_intermediate,
    modification_name_changes=modification_rules_step2,
    vegan_list=remapping.get("vegan_list", []),
    vegetarian_list=remapping.get("vegetarian_list", []),
    meat_list=remapping.get("meat_list", []),
    alcohol_list=remapping.get("alcoholic_drinks", []),
    drinks_list=remapping.get("non_alcoholic_drinks", []),
    merch=remapping.get("merch_list", []),
    rare=rare_items + remapping.get("rare_list", []),
    unknown=remapping.get("unknown_list", [])
)

df_relabeled.to_parquet(DATA_DIR_3_1 / (loc_id + '_sales_and_menu.parquet'))

# Consolidate dishes
consolidating_names = {dish: [f"Vegan {dish}", f"Vegetarian {dish}", f"Meat {dish}"] for dish, _, _ in DISH_CONFIG}
df_consolidated = df_relabeled.pipe(rename_items, name_changes=consolidating_names)

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, top_n=30)
df_consolidated.to_parquet(DATA_DIR_3_2 / (loc_id + '_sales_and_menu.parquet'))

In [ ]:
items_less_than_2_dollars = (
    df_targeted
    .groupby('item_name')
    ['unit_price']
    .max()
    .to_frame(name='max_unit_price')
    .query('max_unit_price < 200.0')
    .reset_index()
    .query('~item_name.isin(["Pumpkin Bread","Muffin Marionberry Cobbler","Muffin Chocolate"])')
    .item_name
    .tolist()
    )
missed_merch = ["Variable","Pure","Wow","Tlc","Carton",
                "Honey Stick Pack","Bag Of Bagel",
                "Inji"]
missed_drinks = ["Tap","Choco Beans",
                 "Mango Habanero","Concord",
                 "Cardamom","Kyushu","Ninkasi","Honey Cup",
                 "Vive","Strawberry White Chocolate","Pumpkin Pie","Weekly Special"]
take_and_go = ["Kind","Rxbar","Kure Bar"]
cookies = ["Chocolate Chip Cookie","Strawberry Matcha Cookie",
           "Coffee Cake Cookie","Cookie Sampler","Pandan Cookie",
           "House Made Cookie","Cookie","Sea Salt Caramel Nutella"]

modification_name_changes = [
    ('Bagel', 'Egg, Hummus Pesto', 'Egg Hummus Pesto'),
    ('Bagel', '^(?=.*Egg)(?=.*Cheese)(?=.*Meat).*', 'Egg Meat Cheese'),
    ('Bagel', '^(?=.*Egg)(?=.*Cheese).*', 'Egg Cheese'),
    ('Bagel', '^(?=.*Egg)(?=.*Meat).*', 'Egg Meat'),
    ('Egg Sandwich', '4. Tom Swanson Plain', 'Tom Swanson'),
    ('Egg Sandwich','Egusto Mato', 'Egusto Mato'),
    ('Egg Sandwich','Egusto Swanson', 'Egusto Swanson'),
    ('Egg Sandwich','5. Egusto', 'Egusto'),
    ('Egg Sandwich', '^(?=.*Egg)(?=.*Cheese)(?=.*Meat).*', 'Egg Meat Cheese'),
    ('Egg Sandwich','Egg Cheese', 'Egg Cheese'),
    ('Sandwich - Breakfast','2','Tomwich'),
    ('Sandwich - Breakfast','3|^(?=.*Egg)(?=.*Meat)(?=.*Cheese)(?!.*Bagel).*','Egg Meat Cheese'),
    ('Sandwich - Breakfast','^(?=.*Egg)(?=.*Meat)(?!.*Bagel).*','Egg Meat'),
    ('Sandwich - Breakfast','1','Egg Cheese'),
    ('Sandwich - Breakfast','4','Tom Swanson'),
    ('Sandwich - Breakfast','5','Egusto'),
    ('Sandwich - Breakfast','6','Egusto Mato'),
    ('Sandwich - Breakfast','7','Egusto Swanson'),
    ('Sandwich - Breakfast','8|Griffith St','Griffith Street'),
    ('Sandwich - Breakfast','','Bagel'),
    ('Whole Grain','Egg, Hummus Pesto','Egg Hummus Pesto'),
    ('Whole Grain','^(?=.*Egg)(?=.*Cheese)(?=.*Meat).*','Egg Meat Cheese'),
    ('Whole Grain','^(?=.*Egg)(?=.*Cheese).*','Egg Cheese'),
    ('Whole Grain','^(?=.*Egg)(?=.*Meat).*','Egg Meat'),
    ('Whole Grain','','Bagel'),
    ('Little Little Loaf','Banana','Banana Loaf'),
    ('Little Little Loaf','Pumpkin','Pumpkin Loaf'),
    ('Little Little Loaf','Matcha','Matcha Loaf')
]

name_changes = {
    'Danish Blueberry':['Danish - Blueberry'],
    'The Jesse Sandwich':['Vegan The Jesse Sandwich'],
    'Turkey Sandwich With Side Salad':['Turkey Sandwich'],
    'Chicken Salad Sandwich With Side Salad':['Chicken Salad Sandwich'],
    'Hummus Plate':['Hummus','Veggie Hummus'],
    'Fullmetal Alchemist': ['Full Metal Alchemist'],
    'Simple Salad':['Salad'],
    'Scone Vanilla Bean':['Scone - Vanilla Bean','Vanilla Scone'],
    'Egg Cheese':['Egg Cheddar'],
    'Scone Blueberry':['Scone - Blueberry','Blueberry Scone'],
    'Muffin Marionberry':['Muffin - Marionberry','Muffin Marionberry Cobbler'],
    'Danish Lemon':['Danish - Lemon'],
    'Veggie Sandwich':['Veggie']
}

filtered = (
    df_targeted
    .query('~item_name.isin(@items_less_than_2_dollars)')
    .query('~item_name.isin(@missed_merch)')
    .query('~item_name.isin(@missed_drinks)')
    .query('~item_name.isin(@take_and_go)')
    .query('~item_name.isin(@cookies)')
    .pipe(lambda df: rename_items(df, name_changes))
    .pipe(lambda df: rename_items_by_modifications(df, modification_name_changes))
    .query('~item_name.isin(["Little Little Loaf"])'))


print(
    filtered
    .groupby('item_name')
    ['item_modifications']
    .apply(lambda s: s.value_counts().index.str.slice(0,20).tolist()[0:3])
    .loc[filtered.item_name.value_counts().index.tolist()]
    .to_frame()
    .join(filtered.item_name.value_counts())
    .reset_index()
    .set_index('count')
    [['item_name','item_modifications']]
    #.iloc[10:,:]
    .to_string()
)


plot_dish_time_series(filtered.set_index('created_at'), loc_id, before_after_details_true, top_n=70)


In [ ]:
presence_dict = {}
for item, group in filtered.groupby("item_name"):
    presence_dict[item] = infer_active_days(group["created_at"], max_gap_days=120)
presence_df = pd.concat(presence_dict, axis=1).fillna(False)

presence_weekly = presence_df.resample('W').max().fillna(False).astype(bool).to_period('W')
dish_order = list(filtered.item_name.value_counts().index)
plot_boolean_time_series(presence_weekly, loc_id, before_after_details_true, dish_order, [])
plot_dish_time_series(filtered.set_index('created_at'), loc_id, before_after_details_true, top_n=70)

presence_daily = strict_bridge_fill(presence_df, limit=7).resample('D').max()
#presence_daily = (presence_weekly.astype(float).replace(0.0, np.nan)[vegetarian_dishes].resample('D').interpolate(limit=6).fillna(0).astype(int).sum(axis=1).plot())
menu = pd.read_csv(Path("scripts") / "labeling" / "dish_labels" / (loc_id + ".csv"))

vegan_dishes = menu.loc[menu["vegan"], "item_name"]
vegetarian_dishes = menu.loc[menu["vegetarian"], "item_name"]
mpbamod_dishes = menu.loc[menu["mpbamod"], "item_name"]

vegan_dishes_count = presence_daily[vegan_dishes].sum(axis=1)
vegetarian_dishes_count = presence_daily[vegetarian_dishes].sum(axis=1)
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
indicator = pd.to_datetime(promo_date) < presence_daily.index
mpbamod_dishes_count = presence_daily[mpbamod_dishes].sum(axis=1) * indicator
menu_counts = pd.concat([vegan_dishes_count, vegetarian_dishes_count, mpbamod_dishes_count], axis=1).set_axis(['vegan_dishes_count', 'vegetarian_dishes_count', 'mpbamod_dishes_count'], axis=1)
menu_counts.plot()
menu_counts.to_csv(Path("scripts") / "labeling" / "dish_counts" / (loc_id + ".csv"))

In [ ]:
# df_targeted.query('item_name == "Veggie Sandwich"')[['unit_price','item_modifications']].value_counts()
# print(filtered.query('item_modifications.str.contains("Vegan")')[['item_name','item_modifications']].value_counts().to_string())
# print(df_targeted.item_name.value_counts().to_string())
# print(filtered.query('item_name.isin(["Tom Swanson","Tomwich","Egusto Mato","Egusto","Egusto Swanson"])')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').query('item_name.str.contains("Bacon") or item_modifications.str.contains("Bacon")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').query('item_name.str.contains("Bacon") or item_modifications.str.contains("Bacon")')['item_quantity'].sum())

# Parse promo strings / lists
promo_name = before_after_details_true.loc[loc_id,'promo_name']
if isinstance(promo_name, str) and promo_name.startswith('['):
    import ast
    promo_list = ast.literal_eval(promo_name)
    vegan_str, bacon_str = promo_list[0], promo_list[1]
else:
    vegan_str, bacon_str = "Vegan", "Bacon"

# Define filters
filters = {
    'Vegan Bacon': lambda df: (df['item_name'].str.contains(vegan_str, na=False) & df['item_name'].str.contains(bacon_str, na=False)) | (df['item_modifications'].str.contains(vegan_str, na=False) & df['item_modifications'].str.contains(bacon_str, na=False)),
    'Make It Vegan': lambda df: df['item_modifications'].str.contains("Make It Vegan", na=False),
    'Vegan Egg': lambda df: df['item_modifications'].str.contains("Vegan Egg|Just Egg", na=False),
    'Vegan Cream Cheese': lambda df: df['item_modifications'].str.contains("Vegan Cream Cheese", na=False),
    'Vegan Sausage': lambda df: df['item_modifications'].str.contains("Vegan Sausage", na=False),
    'Vegan Cheese': lambda df: df['item_modifications'].str.contains("Vegan Cheese|Vegan Cheddar", na=False)
}

# Plot results
plt.figure(figsize=(12, 6))
for label, filter_func in filters.items():
    data = df_uncleaned.loc[filter_func(df_uncleaned)]
    plt.plot(data.resample('W')['item_quantity'].sum(), label=label)
plt.axvline(x=promo_date, color='red', linestyle='--', label='Promo Date')
plt.title("Plant-Based Analog")
plt.legend()
plt.xticks(rotation=50)
plt.show()